## Basic Information Retrieval System Using the CISI Dataset ##


### Setup ###

In [8]:
import math, re
from collections import defaultdict, Counter
from functools import lru_cache

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords", quiet=True)
DATA_DIR = "archive"

In [9]:
DATA_DIR = r"C:\Users\monica\Documents\11th trimester\Introduction to Information Retrieval\Week6CISISystem\archive"

### Parsing the collection ###

In [3]:
def parse_cisi(path, fields=("T", "W")):
    texts, cur, field = {}, None, None
    with open(path, encoding="latin-1") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I"):
                cur = int(line.split()[1]); texts[cur] = ""; field = None
            elif line[:2] in (".T", ".W"):
                field = line[1]
            elif line[:2] in (".A", ".B", ".X"):
                field = None
            elif field in fields and cur is not None:
                texts[cur] += " " + line
    return texts

def parse_rel(path):
    rel = defaultdict(set)
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 2:
                rel[int(p[0])].add(int(p[1]))
    return rel

docs    = parse_cisi(f"{DATA_DIR}/CISI.ALL")
queries = parse_cisi(f"{DATA_DIR}/CISI.QRY")
rels    = parse_rel(f"{DATA_DIR}/CISI.REL")

print(f"Documents: {len(docs)} | Queries: {len(queries)} | Judged queries: {len(rels)}")
print("\nExample document 1:", docs[1].strip()[:200], "...")
print("\nExample query 1:", queries[1].strip())

Documents: 1460 | Queries: 112 | Judged queries: 76

Example document 1: 18 Editions of the Dewey Decimal Classifications    The present study is a history of the DEWEY Decimal Classification.  The first edition of the DDC was published in 1876, the eighteenth edition in 1 ...

Example query 1: What problems and concerns are there in making up descriptive titles? What difficulties are involved in automatically retrieving articles from approximate titles? What is the usual relevance of the content of articles to their titles?


### Preprocessing ###

In [12]:
STOP = set(stopwords.words("english"))
stemmer = PorterStemmer()

@lru_cache(maxsize=None)
def stem(word):
    return stemmer.stem(word)

def preprocess(text, use_stop=True, use_stem=True):
    tokens = re.findall(r"[a-z]+", text.lower())
    if use_stop:
        tokens = [t for t in tokens if t not in STOP]
    if use_stem:
        tokens = [stem(t) for t in tokens]
    return tokens

sample = "Automatic indexing of library documents is studied."
print("Before:", sample)
print("After: ", preprocess(sample))

Before: Automatic indexing of library documents is studied.
After:  ['automat', 'index', 'librari', 'document', 'studi']


### Indexing ###

In [13]:
class InvertedIndex:
    def __init__(self, use_stop=True, use_stem=True):
        self.use_stop, self.use_stem = use_stop, use_stem
        self.postings = defaultdict(dict)   # term -> {doc_id: term frequency}
        self.doc_len, self.norm = {}, {}
        self.N, self.avgdl = 0, 0.0

    def analyze(self, text):
        return preprocess(text, self.use_stop, self.use_stem)

    def idf(self, term):
        return math.log(self.N / len(self.postings[term]))

    def build(self, docs):
        for doc_id, text in docs.items():
            terms = self.analyze(text)
            self.doc_len[doc_id] = len(terms)
            for term, tf in Counter(terms).items():
                self.postings[term][doc_id] = tf
        self.N = len(self.doc_len)
        self.avgdl = sum(self.doc_len.values()) / self.N
        sq = defaultdict(float)
        for term, plist in self.postings.items():
            idf = self.idf(term)
            for d, tf in plist.items():
                sq[d] += ((1 + math.log(tf)) * idf) ** 2
        self.norm = {d: (math.sqrt(v) or 1.0) for d, v in sq.items()}

index = InvertedIndex()
index.build(docs)

print(f"Vocabulary size: {len(index.postings)}")
print(f"Average document length: {index.avgdl:.1f} terms")
print("Postings for 'index' (first 5):", dict(list(index.postings.get("index", {}).items())[:5]))

Vocabulary size: 5673
Average document length: 71.2 terms
Postings for 'index' (first 5): {6: 1, 26: 1, 30: 1, 34: 5, 37: 2}


## Retrieval models ##
### Model 1: TF-IDF with cosine similarity ###

In [14]:
def search_tfidf(query, index, k=10):
    q_weights = {}
    for term, qtf in Counter(index.analyze(query)).items():
        if term in index.postings:
            q_weights[term] = (1 + math.log(qtf)) * index.idf(term)

    q_norm = math.sqrt(sum(w * w for w in q_weights.values())) or 1.0

    scores = defaultdict(float)
    for term, wq in q_weights.items():
        idf = index.idf(term)
        for d, tf in index.postings[term].items():
            scores[d] += wq * (1 + math.log(tf)) * idf

    ranked = [
        (d, s / (index.norm[d] * q_norm))
        for d, s in scores.items()
    ]

    ranked.sort(key=lambda x: (-x[1], x[0]))
    return ranked[:k]

### Model 2: BM25 ###

In [15]:
def search_bm25(query, index, k=10, k1=1.5, b=0.75):
    scores = defaultdict(float)

    for term in index.analyze(query):
        if term not in index.postings:
            continue

        df = len(index.postings[term])
        idf = math.log(
            1 + (index.N - df + 0.5) / (df + 0.5)
        )

        for d, tf in index.postings[term].items():
            denom = (
                tf + k1 * (
                    1 - b + b * index.doc_len[d] / index.avgdl
                )
            )

            scores[d] += (
                idf * tf * (k1 + 1) / denom
            )

    return sorted(
        scores.items(),
        key=lambda x: (-x[1], x[0])
    )[:k]

### Query processing ###

In [16]:
q = queries[1]
print("Query:", q.strip())
print("Processed terms:", index.analyze(q))

for name, fn in [("TF-IDF", search_tfidf), ("BM25", search_bm25)]:
    print(f"\nTop 5 results ({name}):")
    for d, s in fn(q, index, k=5):
        print(f"  Doc {d:4d} | score {s:.3f} | {docs[d].strip()[:70]}...")

Query: What problems and concerns are there in making up descriptive titles? What difficulties are involved in automatically retrieving articles from approximate titles? What is the usual relevance of the content of articles to their titles?
Processed terms: ['problem', 'concern', 'make', 'descript', 'titl', 'difficulti', 'involv', 'automat', 'retriev', 'articl', 'approxim', 'titl', 'usual', 'relev', 'content', 'articl', 'titl']

Top 5 results (TF-IDF):
  Doc 1281 | score 0.210 | Relative Effectiveness of Document Titles and Abstracts for Determinin...
  Doc  722 | score 0.188 | Information Transfer Limitations of Titles of Chemical Documents   Som...
  Doc  429 | score 0.187 | The Information Content of Titles in Engineering Literature    Since m...
  Doc 1299 | score 0.181 | Current Physics Information    A new concept in science communication ...
  Doc 1294 | score 0.166 | Automatic Text Analysis   In this article the principal experiments in...

Top 5 results (BM25):
  Doc  429 | s

### Evaluation ###

In [17]:
def evaluate(search_fn, index, k_cut=10, **kw):
    P, R, AP = [], [], []
    for qid, relevant in rels.items():
        if qid not in queries:
            continue
        ranked = [d for d, _ in search_fn(queries[qid], index, k=index.N, **kw)]
        hits = sum(1 for d in ranked[:k_cut] if d in relevant)
        P.append(hits / k_cut)
        R.append(hits / len(relevant))
        h, s = 0, 0.0
        for i, d in enumerate(ranked, 1):
            if d in relevant:
                h += 1
                s += h / i
        AP.append(s / len(relevant))
    n = len(P)
    return sum(P) / n, sum(R) / n, sum(AP) / n

print(f"{'Model':28s}{'P@10':>8s}{'R@10':>8s}{'MAP':>8s}")
for name, fn in [("TF-IDF (cosine)", search_tfidf),
                 ("BM25 (k1=1.5, b=0.75)", search_bm25)]:
    p, r, m = evaluate(fn, index)
    print(f"{name:28s}{p:8.4f}{r:8.4f}{m:8.4f}")

Model                           P@10    R@10     MAP
TF-IDF (cosine)               0.3553  0.1452  0.2324
BM25 (k1=1.5, b=0.75)         0.3737  0.1519  0.2309


## Results ##
### BM25 parameter tuning ###

In [18]:
best = (0, None)
for k1 in (0.8, 1.2, 1.5, 2.0):
    for b in (0.5, 0.75, 0.9):
        m = evaluate(search_bm25, index, k1=k1, b=b)[2]
        print(f"k1={k1:<4} b={b:<5} MAP={m:.4f}")
        if m > best[0]:
            best = (m, (k1, b))
print(f"\nBest: k1={best[1][0]}, b={best[1][1]}, MAP={best[0]:.4f}")

k1=0.8  b=0.5   MAP=0.2158
k1=0.8  b=0.75  MAP=0.2229
k1=0.8  b=0.9   MAP=0.2233
k1=1.2  b=0.5   MAP=0.2245
k1=1.2  b=0.75  MAP=0.2295
k1=1.2  b=0.9   MAP=0.2290
k1=1.5  b=0.5   MAP=0.2283
k1=1.5  b=0.75  MAP=0.2309
k1=1.5  b=0.9   MAP=0.2301
k1=2.0  b=0.5   MAP=0.2321
k1=2.0  b=0.75  MAP=0.2344
k1=2.0  b=0.9   MAP=0.2349

Best: k1=2.0, b=0.9, MAP=0.2349


### Effect of preprocessing ###

In [19]:
print(f"{'Preprocessing':32s}{'MAP (BM25)':>12s}")
for label, us, ust in [("stop-words + stemming", True, True),
                       ("stop-words only", True, False),
                       ("stemming only", False, True),
                       ("no stop-words, no stemming", False, False)]:
    idx = InvertedIndex(use_stop=us, use_stem=ust)
    idx.build(docs)
    print(f"{label:32s}{evaluate(search_bm25, idx)[2]:12.4f}")

Preprocessing                     MAP (BM25)
stop-words + stemming                 0.2309
stop-words only                       0.2088
stemming only                         0.2174
no stop-words, no stemming            0.1929
